In [0]:
%pip install sentence-transformers scikit-learn numpy
dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [0]:
print("Loading embedding model...")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

print("Embedding model loaded.")

In [0]:
INTENT_PROTOTYPES = {

    "legal_qa": [
        "What law applies if I was cheated online?",
        "How do I file a complaint for fraud?",
        "What punishment exists for theft?",
        "What section of law applies to cyber crime?"
    ],

    "scheme_query": [
        "Which government schemes can help poor families?",
        "Am I eligible for any government welfare schemes?",
        "Is there any financial help scheme from government?",
        "Government scheme for education support"
    ],

    "ipc_bns_comparison": [
        "What is the difference between IPC and BNS?",
        "Compare IPC sections with BNS sections",
        "What changed from IPC to Bharatiya Nyaya Sanhita",
        "Explain IPC vs BNS law changes"
    ],

    "legal_summarization": [
        "Summarize this legal section",
        "Explain this law in simple words",
        "Give a short explanation of this legal clause",
        "Simplify this legal document"
    ]

}

In [0]:
print("Precompute prototype embeddings")

protype_embeddings = {}
for intent, ex in INTENT_PROTOTYPES.items():
    formatted_ex = [f"query: {e}" for e in ex]
    embeddings = embedding_model.encode(formatted_ex, normalize_embeddings=True)
    protype_embeddings[intent] = embeddings

print("Prototype embeddings computed")

In [0]:
def sementic_intent_classification(query, threshold = 0.45):
    formatted_query = f"query: {query}"
    intent_scores = {}
    query_embeddings = embedding_model.encode(formatted_query, normalize_embeddings=True).reshape(1, -1)
    for intent, prototypes in protype_embeddings.items():
        cosine_similarities = cosine_similarity(query_embeddings, prototypes)[0]
        intent_scores[intent] = float(np.max(cosine_similarities))

    sorted_scores = sorted(intent_scores.items(), key=lambda x: x[1], reverse=True)
    detected_intents = [
        intent for intent, score in sorted_scores if score >= threshold
    ]

    if len(detected_intents) == 0:
        detected_intents = ["legal_qa"]

    return {
        "query": query,
        "intents": detected_intents,
        "scores": intent_scores
    }

In [0]:
test_queries = [
    "If someone forwards a message in a group that later turns out to cause panic or harm, but they didn’t create it themselves, can they still get into serious trouble, and does it matter whether they knew it was false or not?",

    "can my collage suspend me for conducting an event for taking bribe",

    "Explain BNS section for theft",

    "What changed from IPC to BNS",

    "Summarize this legal section for me"

]

for q in test_queries:
    
    result = sementic_intent_classification(q)

    print("\nQuery:", q)
    print("Detected Intents:", result["intents"])
    print("Scores:", result["scores"])

RAG PIPLINE

In [0]:
%pip install langgraph langchain langchain-community

In [0]:
from typing import TypedDict,List, Dict
from langgraph.graph import StateGraph, END

In [0]:
class AgentState(TypedDict):
    
    query: str
    intents: List[str]
    legal_answer: str
    scheme_answer: str
    comparison_answer: str
    summary_answer: str
    
    final_answer: str

In [0]:
def intent_classifier_node(state: AgentState):

    query = state["query"]
    
    result = sementic_intent_classification(query)
    
    return {
        "intents": result["intents"]
    }

In [0]:
def legal_rag_node(state: AgentState):

    query = state["query"]

    # placeholder for real RAG pipeline
    answer = f"Legal RAG answer for: {query}"

    return {
        "legal_answer": answer
    }
def legal_rag_node(state: AgentState):

    query = state["query"]

    # placeholder for real RAG pipeline
    answer = f"Legal RAG answer for: {query}"

    return {
        "legal_answer": answer
    }

def comparison_node(state: AgentState):

    query = state["query"]

    answer = f"IPC vs BNS comparison result for: {query}"

    return {
        "comparison_answer": answer
    }

def summarization_node(state: AgentState):

    query = state["query"]

    answer = f"Summary of legal text: {query}"

    return {
        "summary_answer": answer
    }

In [0]:
def combine_answers(state: AgentState):

    outputs = []

    if state.get("legal_answer"):
        outputs.append(state["legal_answer"])

    if state.get("scheme_answer"):
        outputs.append(state["scheme_answer"])

    if state.get("comparison_answer"):
        outputs.append(state["comparison_answer"])

    if state.get("summary_answer"):
        outputs.append(state["summary_answer"])

    final = "\n\n".join(outputs)

    return {
        "final_answer": final
    }

In [0]:
def route_by_intent(state: AgentState):

    intents = state["intents"]

    routes = []

    if "legal_qa" in intents:
        routes.append("legal_rag")

    if "scheme_query" in intents:
        routes.append("scheme_rag")

    if "ipc_bns_comparison" in intents:
        routes.append("comparison")

    if "legal_summarization" in intents:
        routes.append("summary")

    return routes

In [0]:
workflow = StateGraph(AgentState)

workflow.add_node("intent_classifier", intent_classifier_node)

workflow.add_node("legal_rag", legal_rag_node)
workflow.add_node("scheme_rag", scheme_rag_node)
workflow.add_node("comparison", comparison_node)
workflow.add_node("summary", summarization_node)

workflow.add_node("combine", combine_answers)

In [0]:
workflow.set_entry_point("intent_classifier")

workflow.add_conditional_edges(
    "intent_classifier",
    route_by_intent,
    {
        "legal_rag": "legal_rag",
        "scheme_rag": "scheme_rag",
        "comparison": "comparison",
        "summary": "summary"
    }
)

workflow.add_edge("legal_rag", "combine")
workflow.add_edge("scheme_rag", "combine")
workflow.add_edge("comparison", "combine")
workflow.add_edge("summary", "combine")

workflow.add_edge("combine", END)

In [0]:
app = workflow.compile()

In [0]:
result = app.invoke({

    "query": "I was cheated in an online payment what law applies and is there any government scheme for help"

})

print(result["final_answer"])

RETRIVAL

In [0]:
%pip install sentence-transformers faiss-cpu transformers accelerate scikit-learn
dbutils.library.restartPython()

In [0]:
import re
import json
import numpy as np
import pandas as pd
import faiss
from typing import List, Dict, TypedDict
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [0]:
# Databricks notebook source
# MAGIC %pip install sentence-transformers scikit-learn numpy
# MAGIC dbutils.library.restartPython()

# COMMAND ----------

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# COMMAND ----------

print("Loading embedding model...")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

print("Embedding model loaded.")

# COMMAND ----------

INTENT_PROTOTYPES = {

    "legal_qa": [
        "What law applies if I was cheated online?",
        "How do I file a complaint for fraud?",
        "What punishment exists for theft?",
        "What section of law applies to cyber crime?"
    ],

    "scheme_query": [
        "Which government schemes can help poor families?",
        "Am I eligible for any government welfare schemes?",
        "Is there any financial help scheme from government?",
        "Government scheme for education support"
    ],

    "ipc_bns_comparison": [
        "What is the difference between IPC and BNS?",
        "Compare IPC sections with BNS sections",
        "What changed from IPC to Bharatiya Nyaya Sanhita",
        "Explain IPC vs BNS law changes"
    ],

    "legal_summarization": [
        "Summarize this legal section",
        "Explain this law in simple words",
        "Give a short explanation of this legal clause",
        "Simplify this legal document"
    ]

}

# COMMAND ----------

print("Precompute prototype embeddings")

protype_embeddings = {}
for intent, ex in INTENT_PROTOTYPES.items():
    formatted_ex = [f"query: {e}" for e in ex]
    embeddings = embedding_model.encode(formatted_ex, normalize_embeddings=True)
    protype_embeddings[intent] = embeddings

print("Prototype embeddings computed")

# COMMAND ----------

def sementic_intent_classification(query, threshold = 0.45):
    formatted_query = f"query: {query}"
    intent_scores = {}
    query_embeddings = embedding_model.encode(formatted_query, normalize_embeddings=True).reshape(1, -1)
    for intent, prototypes in protype_embeddings.items():
        cosine_similarities = cosine_similarity(query_embeddings, prototypes)[0]
        intent_scores[intent] = float(np.max(cosine_similarities))

    sorted_scores = sorted(intent_scores.items(), key=lambda x: x[1], reverse=True)
    detected_intents = [
        intent for intent, score in sorted_scores if score >= threshold
    ]

    if len(detected_intents) == 0:
        detected_intents = ["legal_qa"]

    return {
        "query": query,
        "intents": detected_intents,
        "scores": intent_scores
    }

# COMMAND ----------

test_queries = [
    "If someone forwards a message in a group that later turns out to cause panic or harm, but they didn’t create it themselves, can they still get into serious trouble, and does it matter whether they knew it was false or not?",

    "can my collage suspend me for conducting an event for taking bribe",

    "Explain BNS section for theft",

    "What changed from IPC to BNS",

    "Summarize this legal section for me"

]

for q in test_queries:
    
    result = sementic_intent_classification(q)

    print("\nQuery:", q)
    print("Detected Intents:", result["intents"])
    print("Scores:", result["scores"])

# COMMAND ----------

# MAGIC %md
# MAGIC RAG PIPLINE

# COMMAND ----------

# MAGIC %pip install langgraph langchain langchain-community

# COMMAND ----------

from typing import TypedDict,List, Dict
from langgraph.graph import StateGraph, END

# COMMAND ----------

class AgentState(TypedDict):
    
    query: str
    intents: List[str]
    legal_answer: str
    scheme_answer: str
    comparison_answer: str
    summary_answer: str
    
    final_answer: str

# COMMAND ----------

def intent_classifier_node(state: AgentState):

    query = state["query"]
    
    result = sementic_intent_classification(query)
    
    return {
        "intents": result["intents"]
    }

# COMMAND ----------

def legal_rag_node(state: AgentState):

    query = state["query"]

    # placeholder for real RAG pipeline
    answer = f"Legal RAG answer for: {query}"

    return {
        "legal_answer": answer
    }
def legal_rag_node(state: AgentState):

    query = state["query"]

    # placeholder for real RAG pipeline
    answer = f"Legal RAG answer for: {query}"

    return {
        "legal_answer": answer
    }

def comparison_node(state: AgentState):

    query = state["query"]

    answer = f"IPC vs BNS comparison result for: {query}"

    return {
        "comparison_answer": answer
    }

def summarization_node(state: AgentState):

    query = state["query"]

    answer = f"Summary of legal text: {query}"

    return {
        "summary_answer": answer
    }

# COMMAND ----------

def combine_answers(state: AgentState):

    outputs = []

    if state.get("legal_answer"):
        outputs.append(state["legal_answer"])

    if state.get("scheme_answer"):
        outputs.append(state["scheme_answer"])

    if state.get("comparison_answer"):
        outputs.append(state["comparison_answer"])

    if state.get("summary_answer"):
        outputs.append(state["summary_answer"])

    final = "\n\n".join(outputs)

    return {
        "final_answer": final
    }

# COMMAND ----------

def route_by_intent(state: AgentState):

    intents = state["intents"]

    routes = []

    if "legal_qa" in intents:
        routes.append("legal_rag")

    if "scheme_query" in intents:
        routes.append("scheme_rag")

    if "ipc_bns_comparison" in intents:
        routes.append("comparison")

    if "legal_summarization" in intents:
        routes.append("summary")

    return routes

# COMMAND ----------

workflow = StateGraph(AgentState)

workflow.add_node("intent_classifier", intent_classifier_node)

workflow.add_node("legal_rag", legal_rag_node)
workflow.add_node("scheme_rag", scheme_rag_node)
workflow.add_node("comparison", comparison_node)
workflow.add_node("summary", summarization_node)

workflow.add_node("combine", combine_answers)

# COMMAND ----------

workflow.set_entry_point("intent_classifier")

workflow.add_conditional_edges(
    "intent_classifier",
    route_by_intent,
    {
        "legal_rag": "legal_rag",
        "scheme_rag": "scheme_rag",
        "comparison": "comparison",
        "summary": "summary"
    }
)

workflow.add_edge("legal_rag", "combine")
workflow.add_edge("scheme_rag", "combine")
workflow.add_edge("comparison", "combine")
workflow.add_edge("summary", "combine")

workflow.add_edge("combine", END)

# COMMAND ----------

app = workflow.compile()

# COMMAND ----------

# DBTITLE 1,Cell 19
result = app.invoke({

    "query": "I was cheated in an online payment what law applies and is there any government scheme for help"

})

print(result["final_answer"])

# COMMAND ----------

# MAGIC %md
# MAGIC RETRIVAL

# COMMAND ----------

# MAGIC %pip install sentence-transformers faiss-cpu transformers accelerate scikit-learn
# MAGIC dbutils.library.restartPython()

# COMMAND ----------

import re
import json
import numpy as np
import pandas as pd
import faiss
from typing import List, Dict, TypedDict
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# COMMAND ----------

